# Governance memo back-test (Class 18)

**Braxton Turner & Teenna Pachika** · INFO 4340 Final Project: LLM Spoiler Detection on IMDb Reviews

Each threshold in our governance memo, tested on our 100 hand-labeled reviews: what share of reviews the rule
flags and how many of Claude prompt C's actual errors it catches. It is the code for our governance memo
(final project component 3). Nothing here calls the API: it reads the saved answers in `outputs/`.

## Setup: answer key, saved Claude answers, TF-IDF baseline

In [1]:
import json
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support

SEED = 42
N_BOOT = 1000
rng = np.random.default_rng(SEED)

val = pd.read_csv("data/validation_sample.csv")
train = pd.read_csv("data/train_sample.csv")
key = pd.read_csv("labeling/answer_key.csv").set_index("review_id").loc[val["review_id"]]
true_labels = key["final_label"].to_numpy()
genres = val.assign(genre_one=val["genre"].str.split(", ")).explode("genre_one")


def parse_call(answer):
    """Pull Yes / No / Unsure out of 'Spoiler: ...'; anything else is a parse error."""
    match = re.search(r"Spoiler:\s*\**\s*(Yes|No|Unsure)", str(answer), re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return "parse_error"


def to_flag(pred_labels):
    """Unsure -> Yes (hidden until a human checks); anything else that isn't Yes -> No."""
    return np.array(["Yes" if p in ["Yes", "Unsure"] else "No" for p in pred_labels])


def evaluate(true_labels, pred_labels):
    flags = to_flag(pred_labels)
    p, r, f1, _ = precision_recall_fscore_support(true_labels, flags, labels=["Yes"], zero_division=0)
    return {"recall": r[0], "precision": p[0], "f1": f1[0]}


# Same TF-IDF baseline as Part 4b of the main notebook
vectorizer = TfidfVectorizer(max_features=500, stop_words="english")
X = vectorizer.fit_transform(train["review_text"].tolist())
clf = LogisticRegression(max_iter=1000, random_state=42, C=1.0, class_weight="balanced")
clf.fit(X, np.where(train["is_spoiler"], "Yes", "No"))

# Saved prompt C answers (Teenna's paid run) and their cost
llm_df = pd.DataFrame(json.load(open("outputs/llm_outputs.json")))
best = "LLM prompt C"
c_rows = llm_df[llm_df["prompt"] == "C"].set_index("review_id").loc[val["review_id"]]
preds = pd.DataFrame({best: c_rows["answer"].map(parse_call).to_numpy()}, index=val["review_id"])
cost_per_text = ((c_rows["input_tokens"] * 2.00 + c_rows["output_tokens"] * 10.00) / 1e6).mean()
boot_idx = rng.integers(0, len(val), size=(N_BOOT, len(val)))
print(f"{len(val)} reviews, {(true_labels == 'Yes').sum()} spoilers in our answer key")

100 reviews, 45 spoilers in our answer key


## Back-test: one block per rule

In [2]:
print("=" * 60)
print("GOVERNANCE MEMO BACK-TEST")
print("=" * 60)
print()

# ---- thresholds from our governance memo (change them here, rerun, and the evidence lines update) ----
REVIEW_THRESHOLD = 0.50      # Rule 1: Claude says No but TF-IDF P(spoiler) >= this -> a human checks it
AUDIT_RATE = 0.05            # Rule 1 add-on: random audit of the No calls that are left
RECALL_GATE = 0.80           # Rule 1: system recall must stay at or above this
FAIR_RATIO = 0.80            # Rule 2: four-fifths rule on recall across genres
MIN_SPOILERS_BINDING = 10    # Rule 2 binds for genres with this many spoilers; 5-9 = watch list
MAX_FLIP_RATE = 0.10         # Rule 3: share of calls that may change when the same reviews are asked again
COST_CAP_PER_1000 = 5.00     # Rule 3 add-on: API dollars per 1,000 reviews

p_tfidf = clf.predict_proba(vectorizer.transform(val["review_text"].tolist()))[:, list(clf.classes_).index("Yes")]
claude = to_flag(preds[best].to_numpy())
claude_no = claude == "No"
errors = claude != true_labels
misses = claude_no & (true_labels == "Yes")
print(f"{best} alone: {errors.sum()} errors on {len(val)} reviews "
      f"({misses.sum()} missed spoilers, {errors.sum() - misses.sum()} false flags)")
print(f"Claude said No on {claude_no.sum()} reviews; {misses.sum()} of them ({misses.sum() / claude_no.sum():.0%}) were spoilers\n")


def system_calls(threshold):
    """Claude's call, except No + TF-IDF >= threshold goes to a human (who gets it right)."""
    to_human = claude_no & (p_tfidf >= threshold)
    return to_human, np.where(to_human, true_labels, claude)


backtest = []

# ------------------------------------------------------------ Rule 1: missed spoilers (accuracy)
print("### Rule 1: second opinion on Claude's No calls ###")
sweep = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    to_human, final = system_calls(t)
    sweep.append({"TF-IDF threshold": t, "share flagged": f"{to_human.mean():.0%}",
                  "errors caught": f"{(to_human & errors).sum()} of {errors.sum()}",
                  "misses caught": f"{(to_human & misses).sum()} of {misses.sum()}",
                  "system recall": round(evaluate(true_labels, final)["recall"], 3)})
print(pd.DataFrame(sweep).to_string(index=False))

to_human, final = system_calls(REVIEW_THRESHOLD)
sys_recall = evaluate(true_labels, final)["recall"]
boot_sys = np.array([evaluate(true_labels[b], final[b])["recall"] for b in boot_idx])
lo, hi = np.percentile(boot_sys, [2.5, 97.5])
left_no = claude_no & ~to_human
left_misses = (left_no & misses).sum()
print(f"\nAt {REVIEW_THRESHOLD}: flags {to_human.mean():.0%} of reviews, catches {(to_human & misses).sum()} of "
      f"{misses.sum()} misses. System recall {sys_recall:.2f} [{lo:.2f}, {hi:.2f}] vs. {best} alone "
      f"{evaluate(true_labels, claude)['recall']:.2f}")
print(f"A {AUDIT_RATE:.0%} random audit of the {left_no.sum()} No calls left would catch about "
      f"{AUDIT_RATE * left_misses:.1f} of the {left_misses} misses that remain: it measures the miss rate, it doesn't fix it")
print("Meets the recall gate" if sys_recall >= RECALL_GATE else "BREACH: system recall is below the gate")
if lo < RECALL_GATE:
    print(f"Only 100 reviews: the CI's lower bound ({lo:.2f}) is under {RECALL_GATE}, so confirm on a new labeled batch")
backtest.append({"rule": "1 second opinion", "threshold": f"TF-IDF >= {REVIEW_THRESHOLD}",
                 "share flagged": round(to_human.mean(), 3), "errors caught": int((to_human & errors).sum()),
                 "of errors": int(errors.sum()), "result": f"system recall {sys_recall:.2f} [{lo:.2f}, {hi:.2f}]"})

# ------------------------------------------------------------ Rule 2: fairness across genres
print("\n### Rule 2: four-fifths rule on genre recall ###")
truth_by_id = pd.Series(true_labels, index=val["review_id"])


def genre_table(pred_labels):
    pred_by_id = pd.Series(pred_labels, index=val["review_id"])
    rows = []
    for g, grp in genres.groupby("genre_one"):
        t = truth_by_id[grp["review_id"]].to_numpy()
        if (t == "Yes").sum() >= 5:                      # same cutoff as Part 8b
            rows.append({"genre": g, "n_spoilers": int((t == "Yes").sum()),
                         "recall": round(evaluate(t, pred_by_id[grp["review_id"]].values)["recall"], 3)})
    table = pd.DataFrame(rows)
    return table, table["recall"].min() / table["recall"].max()


for name, calls in [(f"{best} alone", claude), ("with Rule 1", final)]:
    table, ratio = genre_table(calls)
    failing = table.loc[table["recall"] < FAIR_RATIO * table["recall"].max(), "genre"].tolist()
    print(f"{name}: ratio {ratio:.2f}  ({'PASS' if ratio >= FAIR_RATIO else 'BREACH'})  "
          f"genres under {FAIR_RATIO} x best: {', '.join(failing) if failing else 'none'}")
    big = table[table["n_spoilers"] >= MIN_SPOILERS_BINDING]
    big_ratio = big["recall"].min() / big["recall"].max()
    print(f"  genres with {MIN_SPOILERS_BINDING}+ spoilers ({len(big)}): ratio {big_ratio:.2f}  "
          f"({'PASS' if big_ratio >= FAIR_RATIO - 1e-9 else 'BREACH'})")
    if name == f"{best} alone":
        in_failing = val["review_id"].isin(genres.loc[genres["genre_one"].isin(failing), "review_id"]).to_numpy()
        print(f"  Those genres hold {in_failing.mean():.0%} of reviews and {(in_failing & errors).sum()} of "
              f"{errors.sum()} errors")
        backtest.append({"rule": "2 genre fairness", "threshold": f"ratio >= {FAIR_RATIO}",
                         "share flagged": round(in_failing.mean(), 3),
                         "errors caught": int((in_failing & errors).sum()), "of errors": int(errors.sum()),
                         "result": f"ratio {ratio:.2f} alone ({big_ratio:.2f} at {MIN_SPOILERS_BINDING}+)"})
    else:
        backtest[-1]["result"] += f", {ratio:.2f} with Rule 1 ({big_ratio:.2f} at {MIN_SPOILERS_BINDING}+ spoilers)"
print()
print("Genre recall with Rule 1:")
print(table.to_string(index=False))
print(f"Genre groups hold only {table['n_spoilers'].min()}-{table['n_spoilers'].max()} spoilers each: "
      "a low ratio is a warning to relabel and watch, not proof")

# ------------------------------------------------------------ Rule 3: stability (and cost)
print("\n### Rule 3: same reviews, asked again ###")
rerun = {r["review_id"]: parse_call(r["answer"]) for r in json.load(open("outputs/rubric_answers_25.json"))
         if r["prompt"] == best[-1]}
rerun_ids = list(rerun)
first_call = pd.Series(claude, index=val["review_id"])
flipped = [rid for rid in rerun_ids if to_flag([rerun[rid]])[0] != first_call[rid]]
flip_rate = len(flipped) / len(rerun_ids)
pos = val["review_id"].isin(flipped).to_numpy()
print(f"{len(flipped)} of {len(rerun_ids)} calls changed ({flip_rate:.0%}; limit {MAX_FLIP_RATE:.0%}): {', '.join(flipped)}")
print(f"{(pos & errors).sum()} of the {len(flipped)} flipped reviews were errors in the classifier run; "
      f"Rule 1 already sends {(pos & to_human).sum()} of them to a human")
print("BREACH: too many calls change between runs" if flip_rate > MAX_FLIP_RATE else "Stable enough")
api_per_1000 = cost_per_text * 1000
print(f"API cost ${api_per_1000:.2f} per 1,000 reviews (cap ${COST_CAP_PER_1000:.2f}): "
      f"{'OK' if api_per_1000 <= COST_CAP_PER_1000 else 'BREACH'}")
print(f"Human workload from Rule 1: {to_human.mean() * 1000:.0f} reviews per 1,000")
backtest.append({"rule": "3 stability", "threshold": f"flip rate <= {MAX_FLIP_RATE:.0%}",
                 "share flagged": round(flip_rate, 3), "errors caught": int((pos & errors).sum()),
                 "of errors": int(errors.sum()), "result": f"{len(flipped)} of {len(rerun_ids)} flipped"})

backtest = pd.DataFrame(backtest)
print("\n### Back-test summary ###")
print(backtest.to_string(index=False))
backtest.to_csv("outputs/governance_backtest.csv", index=False)
assert backtest["errors caught"].le(backtest["of errors"]).all(), "a rule caught more errors than exist"

GOVERNANCE MEMO BACK-TEST

LLM prompt C alone: 27 errors on 100 reviews (24 missed spoilers, 3 false flags)
Claude said No on 76 reviews; 24 of them (32%) were spoilers

### Rule 1: second opinion on Claude's No calls ###
 TF-IDF threshold share flagged errors caught misses caught  system recall
              0.3           65%      21 of 27      21 of 24          0.933
              0.4           49%      19 of 27      19 of 24          0.889
              0.5           39%      17 of 27      17 of 24          0.844
              0.6           18%      10 of 27      10 of 24          0.689
              0.7            9%       7 of 27       7 of 24          0.622



At 0.5: flags 39% of reviews, catches 17 of 24 misses. System recall 0.84 [0.73, 0.94] vs. LLM prompt C alone 0.47
A 5% random audit of the 37 No calls left would catch about 0.4 of the 7 misses that remain: it measures the miss rate, it doesn't fix it
Meets the recall gate
Only 100 reviews: the CI's lower bound (0.73) is under 0.8, so confirm on a new labeled batch

### Rule 2: four-fifths rule on genre recall ###
LLM prompt C alone: ratio 0.27  (BREACH)  genres under 0.8 x best: Adventure, Comedy, Family, Fantasy
  genres with 10+ spoilers (5): ratio 0.46  (BREACH)
  Those genres hold 56% of reviews and 19 of 27 errors
with Rule 1: ratio 0.75  (BREACH)  genres under 0.8 x best: Crime
  genres with 10+ spoilers (5): ratio 0.80  (PASS)

Genre recall with Rule 1:
    genre  n_spoilers  recall
   Action          21   0.857
Adventure          15   0.867
   Comedy          11   0.818
    Crime           8   0.750
    Drama          20   0.800
   Family           5   0.800
  Fantasy       